## 2. Ingest_leagues_data

Lee `leagues.csv` desde el contenedor raw de ADLS, aplica schema,
renombra tipos y escribe `football_dev.bronze.leagues`.


In [0]:
dbutils.widgets.removeAll()


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "football_dev")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlssmartdata1702")


In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/leagues.csv" 


In [0]:
leagues_schema = StructType(fields=[
    StructField("league_id", IntegerType(), False),
    StructField("league_ref", StringType(), True),
    StructField("name", StringType(), True),
    StructField("country", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("confederation", StringType(), True),
    StructField("tier", IntegerType(), True),
    StructField("founded_year", IntegerType(), True)
])


In [0]:
df_leagues = spark.read\
    .option("header", True)\
    .schema(leagues_schema)\
    .csv(ruta)


In [0]:
leagues_final_df = df_leagues.withColumn("ingestion_date", current_timestamp())


In [0]:
leagues_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.leagues")
